# Synthetic drift scenarios

This notebook is an interactive walkthrough of the synthetic scenarios used to evaluate `WassersteinDriftDetector` against the River baselines. It covers abrupt functional changes as well as distribution shifts designed to preserve the mean and variance.

Run it from the repository root after installing the project with `uv sync`. The full reproducible benchmark is available in `experiments/run_all_tests.py`; this notebook is intended for exploration and interpretation rather than automated testing. All random seeds are fixed, and generated outputs are kept out of version control.

In [ ]:
import numpy as np
import pandas as pd

from wasserstein_score.evaluation import calibrate_and_test, pivot_test_results
from wasserstein_score.helper_functions import get_model
from wasserstein_score.normalization import CalibrationMedianNormalizer

In [ ]:
pd.options.plotting.backend = "plotly"

In [ ]:
num_datapoints = 6000
drift_at = 3000

n_calibrations_streams = 5
n_test_streams = 5

# Set True to divide errors by the pooled drift-free calibration median.
rescale_by_calibration_median = False

In [ ]:
from wasserstein_score.evaluation import calibrate_and_test, pivot_test_results
from wasserstein_score.helper_functions import (
    get_model, build_model_df,
    abrupt_linear_drift_stream, friedman_f1_f2_switch, friedman_f2_f1_switch, river_friedman_drift_stream,
    bimodal_shift_streams,
)

running_algos = ["wasserstein", "ph", "kswin", "adwin"]

### Abrupt linear

In [ ]:
X_training, y_training, X_calib_data, y_calib_data, X_test_data, y_test_data = abrupt_linear_drift_stream(
    n=num_datapoints, drift_at=drift_at, noise=0.1, n_test_streams=n_test_streams, n_calibrations_streams=n_calibrations_streams,
)

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['abrupt'][0]))]).reset_index(drop=True).plot()

In [ ]:
model = get_model(
    build_model_df(X_training, y_training),
    3500,
    1500,
    range(10),
    'target'
)

In [ ]:
y_calib_residuals = [
    (model.predict(X_calib_data[i]) - y_calib_data[i]) ** 2
    for i in range(len(X_calib_data))
]
y_test_residuals = {
    name: [
        (model.predict(X_test_data[name][i]) - y_test_data[name][i]) ** 2
        for i in range(len(X_test_data[name]))
    ]
    for name in X_test_data
}

# Fit only on pooled drift-free calibration squared errors; freeze for testing.
normalizer = CalibrationMedianNormalizer.fit(
    y_calib_residuals, enabled=rescale_by_calibration_median
)
y_calib_residuals = normalizer.transform(y_calib_residuals)
y_test_residuals = {
    name: normalizer.transform(streams) for name, streams in y_test_residuals.items()
}
print(f"Signal normalization: {normalizer.method}; divisor: {normalizer.scale:g}")

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals[1])), pd.DataFrame(np.array(y_test_residuals['abrupt'][1]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals).mean(axis=0)), pd.DataFrame(np.array(y_test_residuals['abrupt']).mean(axis=0))]).reset_index(drop=True).plot()

In [ ]:
starting_idx = 0

In [ ]:
best_params, calib_df, test_df, per_stream_df = calibrate_and_test(
    calibration_streams=y_calib_residuals,
    test_streams=y_test_residuals,
    drift_at=drift_at,
    starting_idx=starting_idx,
    algorithms=running_algos,
    verbose=True,
)

pivot_test_results(test_df, "mean_first_delay")

### Friedman change from F1 to F2

In [ ]:
X_training, y_training, X_calib_data, y_calib_data, X_test_data, y_test_data = friedman_f1_f2_switch(
    n=num_datapoints, drift_at=drift_at, noise=0.1, n_test_streams=n_test_streams, n_calibrations_streams=n_calibrations_streams,
)

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['abrupt'][0]))]).reset_index(drop=True).plot()

In [ ]:
model = get_model(
    build_model_df(X_training, y_training),
    3500,
    1500,
    range(5),
    'target'
)

In [ ]:
y_calib_residuals = [
    (model.predict(X_calib_data[i]) - y_calib_data[i]) ** 2
    for i in range(len(X_calib_data))
]
y_test_residuals = {
    name: [
        (model.predict(X_test_data[name][i]) - y_test_data[name][i]) ** 2
        for i in range(len(X_test_data[name]))
    ]
    for name in X_test_data
}

# Fit only on pooled drift-free calibration squared errors; freeze for testing.
normalizer = CalibrationMedianNormalizer.fit(
    y_calib_residuals, enabled=rescale_by_calibration_median
)
y_calib_residuals = normalizer.transform(y_calib_residuals)
y_test_residuals = {
    name: normalizer.transform(streams) for name, streams in y_test_residuals.items()
}
print(f"Signal normalization: {normalizer.method}; divisor: {normalizer.scale:g}")

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals[0])), pd.DataFrame(np.array(y_test_residuals['abrupt'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals).mean(axis=0)), pd.DataFrame(np.array(y_test_residuals['abrupt']).mean(axis=0))]).reset_index(drop=True).plot()

In [ ]:
best_params, calib_df, test_df, per_stream_df = calibrate_and_test(
    calibration_streams=y_calib_residuals,
    test_streams=y_test_residuals,
    drift_at=drift_at,
    starting_idx=starting_idx,
    algorithms=running_algos,
    verbose=True,
)

pivot_test_results(test_df, "mean_first_delay")

### Friedman change from F2 to F1

In [ ]:
X_training, y_training, X_calib_data, y_calib_data, X_test_data, y_test_data = friedman_f2_f1_switch(
    n=num_datapoints, drift_at=drift_at, noise=0.1, n_test_streams=n_test_streams, n_calibrations_streams=n_calibrations_streams,
)

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['abrupt'][0]))]).reset_index(drop=True).plot()

In [ ]:
model = get_model(
    build_model_df(X_training, y_training),
    3500,
    1500,
    range(5),
    'target'
)

In [ ]:
y_calib_residuals = [
    (model.predict(X_calib_data[i]) - y_calib_data[i]) ** 2
    for i in range(len(X_calib_data))
]
y_test_residuals = {
    name: [
        (model.predict(X_test_data[name][i]) - y_test_data[name][i]) ** 2
        for i in range(len(X_test_data[name]))
    ]
    for name in X_test_data
}

# Fit only on pooled drift-free calibration squared errors; freeze for testing.
normalizer = CalibrationMedianNormalizer.fit(
    y_calib_residuals, enabled=rescale_by_calibration_median
)
y_calib_residuals = normalizer.transform(y_calib_residuals)
y_test_residuals = {
    name: normalizer.transform(streams) for name, streams in y_test_residuals.items()
}
print(f"Signal normalization: {normalizer.method}; divisor: {normalizer.scale:g}")

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals[0])), pd.DataFrame(np.array(y_test_residuals['abrupt'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals).mean(axis=0)), pd.DataFrame(np.array(y_test_residuals['abrupt']).mean(axis=0))]).reset_index(drop=True).plot()

In [ ]:
best_params, calib_df, test_df, per_stream_df = calibrate_and_test(
    calibration_streams=y_calib_residuals,
    test_streams=y_test_residuals,
    drift_at=drift_at,
    starting_idx=starting_idx,
    algorithms=running_algos,
    verbose=True,
)

pivot_test_results(test_df, "mean_first_delay")

### River Friedman Dataset

In [ ]:
X_training, y_training, X_calib_data, y_calib_data, X_test_data, y_test_data = river_friedman_drift_stream(
    n=num_datapoints, drift_at=drift_at, n_test_streams=n_test_streams, n_calibrations_streams=n_calibrations_streams,
)

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['abrupt_global_gsg'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['abrupt_local_lea'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['gradual_fast'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['gradual_slow'][0]))]).reset_index(drop=True).plot()

In [ ]:
model = get_model(
    build_model_df(X_training, y_training),
    3500,
    1500,
    range(10),
    'target'
)

In [ ]:
y_calib_residuals = [
    (model.predict(X_calib_data[i]) - y_calib_data[i]) ** 2
    for i in range(len(X_calib_data))
]
y_test_residuals = {
    name: [
        (model.predict(X_test_data[name][i]) - y_test_data[name][i]) ** 2
        for i in range(len(X_test_data[name]))
    ]
    for name in X_test_data
}

# Fit only on pooled drift-free calibration squared errors; freeze for testing.
normalizer = CalibrationMedianNormalizer.fit(
    y_calib_residuals, enabled=rescale_by_calibration_median
)
y_calib_residuals = normalizer.transform(y_calib_residuals)
y_test_residuals = {
    name: normalizer.transform(streams) for name, streams in y_test_residuals.items()
}
print(f"Signal normalization: {normalizer.method}; divisor: {normalizer.scale:g}")

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals[0])), pd.DataFrame(np.array(y_test_residuals['abrupt_global_gsg'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals).mean(axis=0)), pd.DataFrame(np.array(y_test_residuals['abrupt_global_gsg']).mean(axis=0))]).reset_index(drop=True).plot()

In [ ]:
best_params, calib_df, test_df, per_stream_df = calibrate_and_test(
    calibration_streams=y_calib_residuals,
    test_streams=y_test_residuals,
    drift_at=drift_at,
    starting_idx=starting_idx,
    algorithms=running_algos,
    verbose=True,
)

pivot_test_results(test_df, "mean_first_delay")

### Bimodal shift (mean- and variance-preserving)

In [ ]:
X_training, y_training, X_calib_data, y_calib_data, X_test_data, y_test_data = bimodal_shift_streams(
    n=num_datapoints, drift_at=drift_at, base_sigma=1.0, separation=0.9,
    n_test_streams=n_test_streams, n_calibrations_streams=n_calibrations_streams, seed_base=2000,
)

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_data[0])), pd.DataFrame(np.array(y_test_data['bimodal_shift'][0]))]).reset_index(drop=True).plot()

In [ ]:
model = get_model(
    build_model_df(X_training, y_training),
    3500,
    1500,
    range(10),
    'target'
)

In [ ]:
y_calib_residuals = [
    (model.predict(X_calib_data[i]) - y_calib_data[i]) ** 2
    for i in range(len(X_calib_data))
]
y_test_residuals = {
    name: [
        (model.predict(X_test_data[name][i]) - y_test_data[name][i]) ** 2
        for i in range(len(X_test_data[name]))
    ]
    for name in X_test_data
}

# Fit only on pooled drift-free calibration squared errors; freeze for testing.
normalizer = CalibrationMedianNormalizer.fit(
    y_calib_residuals, enabled=rescale_by_calibration_median
)
y_calib_residuals = normalizer.transform(y_calib_residuals)
y_test_residuals = {
    name: normalizer.transform(streams) for name, streams in y_test_residuals.items()
}
print(f"Signal normalization: {normalizer.method}; divisor: {normalizer.scale:g}")

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals[0])), pd.DataFrame(np.array(y_test_residuals['bimodal_shift'][0]))]).reset_index(drop=True).plot()

In [ ]:
pd.concat([pd.DataFrame(np.array(y_calib_residuals).mean(axis=0)), pd.DataFrame(np.array(y_test_residuals['bimodal_shift']).mean(axis=0))]).reset_index(drop=True).plot()

In [ ]:
best_params, calib_df, test_df, per_stream_df = calibrate_and_test(
    calibration_streams=y_calib_residuals,
    test_streams=y_test_residuals,
    drift_at=drift_at,
    starting_idx=starting_idx,
    algorithms=running_algos,
    verbose=True,
)

pivot_test_results(test_df, "mean_first_delay")